## COMPARAÇÃO DE ARQUITETURAS DE REDES NEURAIS CONVOLUCIONAIS PARA CLASSIFICAÇÃO DE CÁRIES EM RADIOGRAFIAS PANORÂMICAS

### v3 — Reformulação metodológica

**Pré-requisito:** dataset já pré-processado e disponível em `DENTAL_PREPROCESSED/`.
As Células 4 e 5 do notebook original não são executadas nesta versão.

**Alterações em relação à v2.1:**
- Backup integral do dataset pré-processado antes de qualquer modificação
- Manifesto existente carregado diretamente do Drive (sem reprocessamento)
- Caminhos de INREDD e DENTEX removidos (não utilizados)
- Divisão 1/3 treino/validação + 2/3 inferência (estratificada por classe)
- Verificação explícita de balanceamento de classes (50%/50%)
- Data augmentation completamente removido
- Verificação robusta de CUDA com monitoramento de memória GPU

**Distribuição esperada (7.304 imagens totais):**
- Treino/Val (1/3): ~2.434 imagens | ~1.217 por classe
- Inferência (2/3): ~4.870 imagens | ~2.435 por classe
- Por fold: ~1.947 treino / ~487 validação

---
## Célula 0 — Backup Integral do Dataset Pré-processado

> **Executar ANTES de qualquer outra operação.**
> Cria cópia completa de `DENTAL_PREPROCESSED/` em `DENTAL_PREPROCESSED_BACKUP/`.
> A célula é idempotente: se o backup já existir, não sobrescreve.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import datetime

OUTPUT_DIR = '/content/drive/MyDrive/DENTAL_PREPROCESSED'
BACKUP_DIR = '/content/drive/MyDrive/DENTAL_PREPROCESSED_BACKUP'

# ── Verificação de pré-condição ───────────────────────────────────────────────
if not os.path.exists(OUTPUT_DIR):
    raise FileNotFoundError(
        f'Diretório de origem não encontrado: {OUTPUT_DIR}\n'
        'Verifique se o dataset pré-processado está disponível no Drive.'
    )

total_files_src = sum(len(files) for _, _, files in os.walk(OUTPUT_DIR))
print(f'Origem  : {OUTPUT_DIR}')
print(f'Destino : {BACKUP_DIR}')
print(f'Arquivos na origem: {total_files_src}')

if os.path.exists(BACKUP_DIR):
    total_files_bk = sum(len(files) for _, _, files in os.walk(BACKUP_DIR))
    print(f'\n[AVISO] Backup já existe com {total_files_bk} arquivos.')
    print('  Nenhuma ação tomada — backup preservado intacto.')
    print('  Para forçar novo backup, apague DENTAL_PREPROCESSED_BACKUP manualmente.')
else:
    print('\nIniciando cópia...')
    t_start = datetime.datetime.now()
    shutil.copytree(OUTPUT_DIR, BACKUP_DIR)
    t_end   = datetime.datetime.now()

    total_files_bk = sum(len(files) for _, _, files in os.walk(BACKUP_DIR))
    status = 'OK' if total_files_bk == total_files_src else (
        f'DIVERGENCIA ({total_files_src} src vs {total_files_bk} bk)'
    )

    print(f'\nBackup concluído em {(t_end - t_start).seconds}s')
    print(f'  Arquivos copiados : {total_files_bk}')
    print(f'  Integridade       : {status}')
    print(f'  Timestamp         : {t_end.strftime("%Y-%m-%d %H:%M:%S")}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CAV_DIR    = '/content/drive/MyDrive/DENTAL_PREPROCESSED/task4_caries/cavitated'
NONCAV_DIR = '/content/drive/MyDrive/DENTAL_PREPROCESSED/task4_caries/non_cavitated'

EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

def contar_imagens(pasta):
    return len([f for f in os.listdir(pasta) if os.path.splitext(f)[1].lower() in EXTENSOES])

cav    = contar_imagens(CAV_DIR)
noncav = contar_imagens(NONCAV_DIR)
total  = cav + noncav

print(f'Cariados     : {cav}')
print(f'Nao-cariados : {noncav}')
print(f'Total        : {total}')
print(f'Balanceado   : {"SIM" if cav == noncav else "NAO"}')

---
## Célula 1 — Configurar Caminhos

> Apenas caminhos necessários para esta versão.
> Caminhos de INREDD e DENTEX removidos — dataset já está pré-processado.

In [ ]:
import os

# ── Diretório principal ───────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/DENTAL_PREPROCESSED'
BACKUP_DIR = '/content/drive/MyDrive/DENTAL_PREPROCESSED_BACKUP'

# ── Splits: separados por finalidade ─────────────────────────────────────────
SPLITS_DIR       = os.path.join(OUTPUT_DIR, 'splits')
SPLITS_TRAIN_DIR = os.path.join(OUTPUT_DIR, 'splits', 'train_val')   # 1/3
SPLITS_INFER_DIR = os.path.join(OUTPUT_DIR, 'splits', 'inference')   # 2/3

MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')

# ── Manifesto existente gerado pela v2.1 ──────────────────────────────────────
MANIFEST_V2_PATH = os.path.join(SPLITS_DIR, 'task4_manifest.json')

# ── Verificar existência do manifesto antes de continuar ──────────────────────
if not os.path.exists(MANIFEST_V2_PATH):
    raise FileNotFoundError(
        f'Manifesto não encontrado: {MANIFEST_V2_PATH}\n'
        'Verifique se task4_manifest.json existe em splits/.'
    )

# ── Criar apenas as pastas novas (existentes são preservadas) ─────────────────
for d in [SPLITS_TRAIN_DIR, SPLITS_INFER_DIR, os.path.join(MODELS_DIR, 'task4')]:
    os.makedirs(d, exist_ok=True)

print('Caminhos configurados:')
print(f'  Output principal : {OUTPUT_DIR}')
print(f'  Backup           : {BACKUP_DIR}')
print(f'  Manifesto v2.1   : {MANIFEST_V2_PATH}')
print(f'  Splits treino/val: {SPLITS_TRAIN_DIR}')
print(f'  Splits inferência: {SPLITS_INFER_DIR}')

---
## Célula 2 — Verificação Robusta de CUDA e GPU

In [ ]:
import subprocess
import tensorflow as tf

print('=' * 60)
print('  VERIFICAÇÃO CUDA / GPU')
print('=' * 60)

# ── 1. nvidia-smi ─────────────────────────────────────────────────────────────
print('\n[1] Driver NVIDIA (nvidia-smi):')
try:
    smi_out = subprocess.check_output(
        ['nvidia-smi',
         '--query-gpu=name,driver_version,memory.total,memory.free,memory.used,temperature.gpu',
         '--format=csv,noheader,nounits'],
        stderr=subprocess.STDOUT, timeout=10
    ).decode().strip()

    for i, linha in enumerate(smi_out.split('\n')):
        campos = [c.strip() for c in linha.split(',')]
        print(f'  GPU {i}: {campos[0]}')
        print(f'    Driver        : {campos[1]}')
        print(f'    Memória total : {campos[2]} MB')
        print(f'    Memória livre : {campos[3]} MB')
        print(f'    Memória usada : {campos[4]} MB')
        print(f'    Temperatura   : {campos[5]} °C')
    NVIDIA_SMI_OK = True
except (subprocess.CalledProcessError, FileNotFoundError, subprocess.TimeoutExpired) as e:
    print(f'  [FALHA] nvidia-smi indisponível: {e}')
    NVIDIA_SMI_OK = False

# ── 2. Detecção lógica de GPU ─────────────────────────────────────────────────
print(f'\n[2] TensorFlow {tf.__version__}:')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'  GPUs detectadas pelo TF : {len(gpus)}')
    for g in gpus:
        print(f'    {g.name}')
    GPU_TF_OK = True
else:
    print('  [AVISO] TensorFlow não detectou nenhuma GPU.')
    GPU_TF_OK = False

# ── 3. Confirmação de execução real em GPU ────────────────────────────────────
print('\n[3] Confirmação de execução em GPU:')
try:
    with tf.device('/GPU:0'):
        c = tf.matmul(tf.constant([[1.0, 2.0], [3.0, 4.0]]),
                      tf.constant([[5.0, 6.0], [7.0, 8.0]]))
    GPU_EXEC_OK = 'GPU' in c.device
    print(f'  Operação executada em: {c.device}')
    print(f'  Execução em GPU: {"CONFIRMADA" if GPU_EXEC_OK else "NAO CONFIRMADA"}')
except RuntimeError as e:
    print(f'  [FALHA] {e}')
    GPU_EXEC_OK = False

# ── 4. Monitoramento de memória CUDA ──────────────────────────────────────────
print('\n[4] Memória CUDA (TensorFlow):')
try:
    mem_info = tf.config.experimental.get_memory_info('GPU:0')
    print(f'  Memória atual (TF) : {mem_info["current"] / 1024**2:.1f} MB')
    print(f'  Pico de memória    : {mem_info["peak"]    / 1024**2:.1f} MB')
    GPU_MEM_OK = True
except Exception as e:
    print(f'  [AVISO] get_memory_info indisponível: {e}')
    GPU_MEM_OK = False

# ── 5. Build info ─────────────────────────────────────────────────────────────
print('\n[5] Build CUDA do TensorFlow:')
print(f'  CUDA compilado : {tf.test.is_built_with_cuda()}')
try:
    print(f'  CUDA versão    : {tf.sysconfig.get_build_info()["cuda_version"]}')
    print(f'  cuDNN versão   : {tf.sysconfig.get_build_info()["cudnn_version"]}')
except Exception:
    pass

# ── Resumo ────────────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  RESUMO')
print('=' * 60)
s = {True: 'OK', False: 'FALHA'}
print(f'  Driver NVIDIA (nvidia-smi) : {s[NVIDIA_SMI_OK]}')
print(f'  GPU detectada pelo TF      : {s[GPU_TF_OK]}')
print(f'  Execução confirmada em GPU : {s[GPU_EXEC_OK]}')
print(f'  Monitoramento de memória   : {s[GPU_MEM_OK]}')

GPU_MONITOR = GPU_MEM_OK

if GPU_TF_OK and GPU_EXEC_OK:
    print('\n  CUDA ativo e funcional. Treinamento será executado em GPU.')
else:
    print('\n  [ATENCAO] GPU não confirmada.')
    print('  Runtime > Alterar tipo de hardware > GPU')
    raise RuntimeError('GPU não detectada — interrompendo para evitar treinamento em CPU.')

---
## Célula 2b — Instalar Dependências

In [ ]:
!pip install -q pandas scikit-learn pillow psutil

import psutil
print(f'TensorFlow : {tf.__version__}')
print(f'psutil     : {psutil.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

---
## Célula 3 — Importações e Funções Utilitárias

In [ ]:
import json
import time
import random
import datetime
import numpy as np
import pandas as pd
import psutil
import os

from collections import Counter, defaultdict

import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    precision_score, recall_score, accuracy_score,
    f1_score, confusion_matrix
)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


def get_ram_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def get_ram_system_mb():
    vm = psutil.virtual_memory()
    return {
        'total_mb'    : vm.total     / 1024 / 1024,
        'used_mb'     : vm.used      / 1024 / 1024,
        'available_mb': vm.available / 1024 / 1024,
        'percent'     : vm.percent
    }

def get_gpu_mb():
    try:
        info = tf.config.experimental.get_memory_info('GPU:0')
        return {'current_mb': info['current'] / 1024 / 1024,
                'peak_mb'   : info['peak']    / 1024 / 1024}
    except Exception:
        return None

def reset_gpu_peak():
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
    except Exception:
        pass

def fmt_duration(seconds):
    return str(datetime.timedelta(seconds=int(seconds)))


class EpochProfilerCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.epoch_logs   = []
        self._epoch_start = None

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        duration = time.time() - self._epoch_start
        try:
            lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        except Exception:
            lr = None
        self.epoch_logs.append({
            'epoch'      : epoch + 1,
            'duration_s' : round(duration, 2),
            'train_acc'  : round(logs.get('accuracy',     0), 6),
            'train_loss' : round(logs.get('loss',         0), 6),
            'val_acc'    : round(logs.get('val_accuracy', 0), 6),
            'val_loss'   : round(logs.get('val_loss',     0), 6),
            'lr'         : lr,
            'ram_proc_mb': round(get_ram_mb(), 1),
            'gpu'        : get_gpu_mb()
        })


print('Importações, callbacks e funções de monitoramento carregados.')

---
## Célula 4 — Carregar Manifesto Existente do Drive

> Dataset já pré-processado. O `task4_manifest.json` da v1 é carregado diretamente,
> sem reprocessar nenhuma imagem. Inclui verificação de existência dos arquivos no disco.

In [ ]:
print(f'Carregando manifesto: {MANIFEST_V2_PATH}')

with open(MANIFEST_V2_PATH) as f:
    task4_manifest = json.load(f)

total = len(task4_manifest)
dist  = Counter(item['label'] for item in task4_manifest)

print(f'\nManifesto carregado:')
print(f'  Total de imagens : {total}')
for cls, cnt in sorted(dist.items()):
    print(f'  {cls:<20}: {cnt} ({cnt/total*100:.2f}%)')

# ── Verificar existência das imagens no disco ─────────────────────────────────
print('\nVerificando existência das imagens no disco...')
ausentes = [item['path'] for item in task4_manifest if not os.path.exists(item['path'])]

if ausentes:
    print(f'  [AVISO] {len(ausentes)} imagens não encontradas no disco.')
    for p in ausentes[:5]:
        print(f'    {p}')
    if len(ausentes) > 5:
        print(f'    ... e mais {len(ausentes) - 5} ausentes.')
else:
    print(f'  Todas as {total} imagens encontradas no disco.')

---
## Célula 5 — Divisão 1/3 Treino/Val + 2/3 Inferência e Verificação de Balanceamento

In [ ]:
from sklearn.model_selection import train_test_split


def check_balance(items, nome_particao, tolerancia_pct=2.0):
    """
    Verifica se a partição está balanceada (50%/50%).
    Retorna True se dentro da tolerância.
    """
    counts = Counter(item['label'] for item in items)
    total  = sum(counts.values())
    print(f'  Partição: {nome_particao}  |  Total: {total} imagens')

    all_ok = True
    for cls in sorted(counts):
        cnt    = counts[cls]
        pct    = cnt / total * 100
        desvio = abs(pct - 50.0)
        ok     = desvio <= tolerancia_pct
        status = 'OK' if ok else f'FORA DA TOLERÂNCIA (desvio={desvio:.2f} pp)'
        print(f'    {cls:<20}: {cnt:>5} ({pct:.2f}%)  [{status}]')
        if not ok:
            all_ok = False

    ratio = max(counts.values()) / min(counts.values()) if min(counts.values()) > 0 else float('inf')
    print(f'    Razão max/min : {ratio:.4f}  (ideal = 1.0000)')
    return all_ok


# ── Divisão estratificada 1/3 + 2/3 ──────────────────────────────────────────
labels_manifest = [item['label'] for item in task4_manifest]

trainval_items, infer_items, _, _ = train_test_split(
    task4_manifest,
    labels_manifest,
    test_size=2/3,
    random_state=RANDOM_SEED,
    stratify=labels_manifest
)

print('Divisão 1/3 + 2/3 (estratificada):')
print(f'  Treino/Validação (1/3): {len(trainval_items)} imagens')
print(f'  Inferência       (2/3): {len(infer_items)} imagens')

# ── Verificação de balanceamento ──────────────────────────────────────────────
print('\n' + '=' * 60)
print('  VERIFICAÇÃO DE BALANCEAMENTO DE CLASSES')
print('=' * 60)

print('\n[Manifesto completo]')
ok_full     = check_balance(task4_manifest, 'Manifesto completo')

print('\n[Partição Treino/Validação — 1/3]')
ok_trainval = check_balance(trainval_items, 'Treino/Validação (1/3)')

print('\n[Partição Inferência — 2/3]')
ok_infer    = check_balance(infer_items, 'Inferência (2/3)')

print('\n' + '─' * 60)
if ok_full and ok_trainval and ok_infer:
    print('  RESULTADO: Todas as partições BALANCEADAS. Prosseguir.')
else:
    print('  RESULTADO: DESEQUILÍBRIO detectado — revisar antes de continuar.')

# ── Persistência dos manifestos ───────────────────────────────────────────────
tv_manifest_path  = os.path.join(SPLITS_TRAIN_DIR, 'task4_trainval_manifest.json')
inf_manifest_path = os.path.join(SPLITS_INFER_DIR, 'task4_inference_manifest.json')

with open(tv_manifest_path,  'w') as f: json.dump(trainval_items, f)
with open(inf_manifest_path, 'w') as f: json.dump(infer_items,    f)

print(f'\nManifestos salvos:')
print(f'  {tv_manifest_path}')
print(f'  {inf_manifest_path}')

---
## Célula 6 — Criar Splits de Validação Cruzada (5-Fold) sobre a Partição 1/3

In [ ]:
def create_5fold_splits_stratified(manifest, prefix, splits_dir, random_seed=42):
    items  = manifest.copy()
    labels = [item['label'] for item in items]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)

    print(f'StratifiedKFold(n_splits=5) sobre {len(items)} imagens (1/3 do dataset)')
    print(f'Esperado: ~{int(len(items)*0.8)} treino / ~{int(len(items)*0.2)} val por fold\n')
    print(f'{"Fold":<6} {"N Treino":>10} {"Cav Tr":>8} {"NCav Tr":>8} '
          f'{"N Val":>8} {"Cav Val":>8} {"NCav Val":>9} {"Bal Tr":>8} {"Bal Val":>8}')
    print('─' * 80)

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(items, labels), start=1):
        train_items = [items[i] for i in train_idx]
        val_items   = [items[i] for i in val_idx]

        tr_dist  = Counter(item['label'] for item in train_items)
        val_dist = Counter(item['label'] for item in val_items)

        tr_cav  = tr_dist.get('cavitated', 0)
        tr_ncav = tr_dist.get('non_cavitated', 0)
        vl_cav  = val_dist.get('cavitated', 0)
        vl_ncav = val_dist.get('non_cavitated', 0)
        tr_bal  = tr_cav / len(train_items) * 100
        val_bal = vl_cav / len(val_items)   * 100

        fold_data = {
            'fold'      : fold_idx,
            'n_train'   : len(train_items),
            'n_val'     : len(val_items),
            'train'     : train_items,
            'val'       : val_items,
            'split_meta': {
                'protocol'   : '1_3_trainval__stratified_5fold_80_20',
                'source_size': len(items),
                'seed'       : random_seed,
                'train_pct'  : round(len(train_items) / len(items) * 100, 2),
                'val_pct'    : round(len(val_items)   / len(items) * 100, 2),
                'train_dist' : dict(tr_dist),
                'val_dist'   : dict(val_dist),
            }
        }

        fold_path = os.path.join(splits_dir, f'{prefix}_fold{fold_idx}.json')
        with open(fold_path, 'w') as f:
            json.dump(fold_data, f)

        print(
            f'{fold_idx:<6} {len(train_items):>10} {tr_cav:>8} {tr_ncav:>8} '
            f'{len(val_items):>8} {vl_cav:>8} {vl_ncav:>9} '
            f'{tr_bal:>7.1f}% {val_bal:>7.1f}%'
        )

    print(f'\nSplits salvos: {prefix}_fold1..5.json em {splits_dir}')


# Proteção: carrega do disco se a Célula 5 não estiver na memória
try:
    _ = trainval_items
except NameError:
    with open(tv_manifest_path) as f:
        trainval_items = json.load(f)
    print(f'Manifesto 1/3 carregado do disco: {len(trainval_items)} imagens')

create_5fold_splits_stratified(
    manifest    = trainval_items,
    prefix      = 'task4',
    splits_dir  = SPLITS_TRAIN_DIR,
    random_seed = RANDOM_SEED,
)

### Célula 6b — Verificação de Integridade dos Splits (6 Checagens)

In [ ]:
EXPECTED_FOLDS    = 5
TRAIN_PCT_MIN     = 0.78
TRAIN_PCT_MAX     = 0.82
VAL_PCT_MIN       = 0.18
VAL_PCT_MAX       = 0.22
CLASS_BALANCE_TOL = 0.03

issues             = []
all_items_per_fold = {}

def section(title):
    print(f'\n{"─"*60}\n  {title}\n{"─"*60}')

section('1. Existência dos arquivos JSON')
for fold in range(1, EXPECTED_FOLDS + 1):
    p  = os.path.join(SPLITS_TRAIN_DIR, f'task4_fold{fold}.json')
    ok = os.path.exists(p)
    print(f'  Fold {fold}: {"OK" if ok else "AUSENTE"}')
    if not ok:
        issues.append(f'Fold {fold}: JSON ausente')

section('2. Estrutura dos JSONs')
REQUIRED_KEYS = {'fold', 'n_train', 'n_val', 'train', 'val', 'split_meta'}
for fold in range(1, EXPECTED_FOLDS + 1):
    p = os.path.join(SPLITS_TRAIN_DIR, f'task4_fold{fold}.json')
    if not os.path.exists(p): continue
    with open(p) as f: fd = json.load(f)
    all_items_per_fold[fold] = {'train': fd['train'], 'val': fd['val']}
    missing = REQUIRED_KEYS - set(fd.keys())
    if missing:
        print(f'  Fold {fold}: chaves ausentes — {missing}')
        issues.append(f'Fold {fold}: chaves ausentes {missing}')
    else:
        print(f'  Fold {fold}: OK ({fd["n_train"]} treino, {fd["n_val"]} val)')

section('3. Proporções treino/val (80%/20%)')
for fold, splits in all_items_per_fold.items():
    total   = len(splits['train']) + len(splits['val'])
    tr_pct  = len(splits['train']) / total
    val_pct = len(splits['val'])   / total
    ok      = TRAIN_PCT_MIN <= tr_pct <= TRAIN_PCT_MAX
    print(f'  Fold {fold}: treino={tr_pct:.1%}  val={val_pct:.1%}  [{"OK" if ok else "FORA"}]')
    if not ok:
        issues.append(f'Fold {fold}: proporção incorreta')

section('4. Balanceamento de classes (50%/50%)')
for fold, splits in all_items_per_fold.items():
    fold_ok = True
    for part_name, part_items in [('treino', splits['train']), ('val', splits['val'])]:
        counts = Counter(item['label'] for item in part_items)
        total  = sum(counts.values())
        for cls, cnt in counts.items():
            if abs(cnt/total - 0.5) > CLASS_BALANCE_TOL:
                msg = f'Fold {fold} {part_name} — {cls}: {cnt/total:.1%}'
                print(f'  [AVISO] {msg}')
                issues.append(msg)
                fold_ok = False
    if fold_ok:
        print(f'  Fold {fold}: OK')

section('5. Existência dos arquivos de imagem')
for fold, splits in all_items_per_fold.items():
    missing = [
        item['path']
        for part in ['train', 'val']
        for item in splits[part]
        if not os.path.exists(item['path'])
    ]
    if missing:
        print(f'  Fold {fold}: {len(missing)} arquivos ausentes')
        issues.append(f'Fold {fold}: {len(missing)} imagens ausentes')
    else:
        print(f'  Fold {fold}: OK')

section('6. Duplicatas e vazamento entre splits')
dup_issues = 0
for fold, splits in all_items_per_fold.items():
    leak = {i['path'] for i in splits['train']} & {i['path'] for i in splits['val']}
    if leak:
        print(f'  [AVISO] Fold {fold}: {len(leak)} imagens em treino E val')
        issues.append(f'Fold {fold}: {len(leak)} amostras com vazamento')
        dup_issues += len(leak)

val_fold_count = defaultdict(list)
for k, splits in all_items_per_fold.items():
    for item in splits['val']:
        val_fold_count[item['path']].append(k)
multi_val = [f for f in val_fold_count.values() if len(f) > 1]
if multi_val:
    print(f'  [AVISO] {len(multi_val)} amostras no val de mais de um fold.')
    issues.append(f'{len(multi_val)} amostras no val de múltiplos folds')
elif dup_issues == 0:
    print('  Sem duplicatas ou vazamento. KFold íntegro.')

section('RESULTADO FINAL')
if not issues:
    print('\n  PASSOU — Splits prontos para treinamento.')
else:
    print(f'\n  FALHOU — {len(issues)} problema(s):')
    for i, msg in enumerate(issues, 1):
        print(f'    {i:2d}. {msg}')

---
## Célula 7 — Construção dos Modelos

In [ ]:
def build_model(architecture, num_classes, dropout_rate):
    input_shape = (299, 299, 3)
    if architecture == 'inceptionv3':
        base = tf.keras.applications.InceptionV3(
            weights='imagenet', include_top=False, input_shape=input_shape)
    elif architecture == 'inceptionresnetv2':
        base = tf.keras.applications.InceptionResNetV2(
            weights='imagenet', include_top=False, input_shape=input_shape)
    elif architecture == 'xception':
        base = tf.keras.applications.Xception(
            weights='imagenet', include_top=False, input_shape=input_shape)
    elif architecture == 'efficientnetv2s':
        base = tf.keras.applications.EfficientNetV2S(
            weights='imagenet', include_top=False, input_shape=input_shape)
    else:
        raise ValueError(f'Arquitetura desconhecida: {architecture}')

    base.trainable = True
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    output = tf.keras.layers.Dense(
        1 if num_classes == 2 else num_classes,
        activation='sigmoid' if num_classes == 2 else 'softmax'
    )(x)
    return tf.keras.Model(inputs=base.input, outputs=output)


def get_callbacks(monitor='val_loss'):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor, patience=6, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=1, verbose=1, min_lr=1e-7)
    ]

print('Funções de modelo e callbacks definidas.')

---
## Célula 8 — Pipeline de Dados SEM Data Augmentation

Ambos os geradores usam apenas `rescale=1./255` — normaliza pixels de [0, 255] para [0, 1].

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TRAIN_DATAGEN = ImageDataGenerator(rescale=1. / 255)
TEST_DATAGEN  = ImageDataGenerator(rescale=1. / 255)

print('ImageDataGenerator configurado SEM augmentation.')


def make_generator(datagen, items, batch_size, class_mode, label_map=None, shuffle=True):
    df = pd.DataFrame(items)
    if class_mode == 'binary':
        label_map = label_map or {'cavitated': '1', 'non_cavitated': '0'}
        df['label_enc'] = df['label'].map(label_map).astype(float)
        y_col, class_mode_gen = 'label_enc', 'raw'
    else:
        y_col, class_mode_gen = 'label', 'sparse'

    return datagen.flow_from_dataframe(
        dataframe=df, x_col='path', y_col=y_col,
        target_size=(299, 299), color_mode='rgb',
        class_mode=class_mode_gen, batch_size=batch_size,
        shuffle=shuffle, seed=RANDOM_SEED
    )

print('Função make_generator carregada.')

---
## Célula 9 — Treinamento Task 4: Detecção de Cáries (5-Fold)

In [ ]:
TASK4_CONFIG = {
    'inceptionv3'      : {'dropout': 0.2, 'lr': 1e-5, 'epochs': 50, 'batch_size': 32},
    'inceptionresnetv2': {'dropout': 0.2, 'lr': 1e-5, 'epochs': 50, 'batch_size': 32},
    'xception'         : {'dropout': 0.2, 'lr': 1e-6, 'epochs': 50, 'batch_size': 32},
    'efficientnetv2s'  : {'dropout': 0.2, 'lr': 1e-5, 'epochs': 50, 'batch_size': 32},
}

T4_LABEL_MAP = {'cavitated': '1', 'non_cavitated': '0'}


def train_task4(architecture):
    cfg       = TASK4_CONFIG[architecture]
    model_dir = os.path.join(MODELS_DIR, 'task4')
    results   = []; profiling = []; history = []

    print(f'\n{"="*60}')
    print(f'TASK 4 | {architecture.upper()} | 5-Fold')
    print(f'  dropout={cfg["dropout"]}  lr={cfg["lr"]}  epochs≤{cfg["epochs"]}  batch={cfg["batch_size"]}')
    #print(f'  Augmentation: DESATIVADO')
    print(f'{"="*60}')

    ram_sys_start  = get_ram_system_mb()
    model_start_ts = datetime.datetime.now().isoformat()
    model_start_t  = time.time()

    for fold in range(1, 6):
        print(f'\n--- Fold {fold}/5 ---')
        reset_gpu_peak()
        ram_before = get_ram_mb()
        gpu_before = get_gpu_mb()
        fold_start = time.time()

        with open(os.path.join(SPLITS_TRAIN_DIR, f'task4_fold{fold}.json')) as f:
            fd = json.load(f)

        train_gen = make_generator(TRAIN_DATAGEN, fd['train'], cfg['batch_size'],
                                   'binary', T4_LABEL_MAP, shuffle=True)
        val_gen   = make_generator(TEST_DATAGEN,  fd['val'],   cfg['batch_size'],
                                   'binary', T4_LABEL_MAP, shuffle=False)

        model = build_model(architecture, num_classes=2, dropout_rate=cfg['dropout'])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
            loss='binary_crossentropy', metrics=['accuracy']
        )

        ckpt_path      = os.path.join(model_dir, f'{architecture}_fold{fold}.h5')
        epoch_profiler = EpochProfilerCallback()
        cbs = get_callbacks() + [
            tf.keras.callbacks.ModelCheckpoint(
                ckpt_path, save_best_only=True, monitor='val_loss', verbose=0),
            epoch_profiler,
        ]

        train_start = time.time()
        model.fit(train_gen, validation_data=val_gen,
                  epochs=cfg['epochs'], callbacks=cbs, verbose=1)
        train_duration = time.time() - train_start

        ram_after_train = get_ram_mb()
        gpu_after_train = get_gpu_mb()
        n_epochs_run    = len(epoch_profiler.epoch_logs)
        best_epoch      = int(np.argmin([e['val_loss'] for e in epoch_profiler.epoch_logs])) + 1

        eval_start   = time.time()
        val_gen_eval = make_generator(TEST_DATAGEN, fd['val'], cfg['batch_size'],
                                      'binary', T4_LABEL_MAP, shuffle=False)
        y_true, y_pred_prob = [], []
        for imgs, labels in val_gen_eval:
            preds = model.predict(imgs, verbose=0).flatten()
            y_pred_prob.extend(preds)
            y_true.extend(labels.astype(int))
            if len(y_true) >= len(fd['val']): break

        eval_duration = time.time() - eval_start
        y_pred = [1 if p >= 0.5 else 0 for p in y_pred_prob]

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(   y_true, y_pred, zero_division=0)
        acc  = accuracy_score( y_true, y_pred)
        f1   = f1_score(       y_true, y_pred, zero_division=0)
        cm   = confusion_matrix(y_true, y_pred)
        spe  = 0.0
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            spe = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        fold_duration  = time.time() - fold_start
        ram_after_eval = get_ram_mb()
        gpu_after_eval = get_gpu_mb()

        results.append({'fold': fold, 'precision': prec, 'recall': rec,
                        'accuracy': acc, 'specificity': spe, 'f1': f1})
        profiling.append({
            'fold': fold, 'n_train_images': len(fd['train']), 'n_val_images': len(fd['val']),
            'epochs_run': n_epochs_run, 'best_epoch': best_epoch,
            'time_train_s': round(train_duration, 2), 'time_eval_s': round(eval_duration, 2),
            'time_fold_total_s': round(fold_duration, 2),
            'time_train_fmt': fmt_duration(train_duration), 'time_fold_fmt': fmt_duration(fold_duration),
            'ram_before_mb': round(ram_before, 1), 'ram_after_train_mb': round(ram_after_train, 1),
            'ram_after_eval_mb': round(ram_after_eval, 1),
            'ram_delta_mb': round(ram_after_eval - ram_before, 1),
            'ram_system_before': ram_sys_start, 'ram_system_after': get_ram_system_mb(),
            'gpu_before': gpu_before, 'gpu_after_train': gpu_after_train, 'gpu_after_eval': gpu_after_eval,
        })
        history.append({'fold': fold, 'epochs': epoch_profiler.epoch_logs})

        print(f'  Fold {fold} → Pre={prec:.4f} Rec={rec:.4f} Acc={acc:.4f} Spe={spe:.4f} F1={f1:.4f}')
        print(f'  Tempo : treino={fmt_duration(train_duration)} | fold={fmt_duration(fold_duration)}')
        if gpu_after_eval:
            print(f'  VRAM  : current={gpu_after_eval["current_mb"]:.0f} MB | peak={gpu_after_eval["peak_mb"]:.0f} MB')

        tf.keras.backend.clear_session()

    model_total_s = time.time() - model_start_t
    model_end_ts  = datetime.datetime.now().isoformat()
    fold_times    = [p['time_fold_total_s'] for p in profiling]

    print(f'\n{"─"*50}')
    print(f'Média 5-fold | {architecture.upper()}')
    for metric in ['precision', 'recall', 'accuracy', 'specificity', 'f1']:
        vals = [r[metric] for r in results]
        print(f'  {metric:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')
    print(f'\n  Tempo total : {fmt_duration(model_total_s)}')
    print(f'  Tempo/fold  : {fmt_duration(np.mean(fold_times))}')

    res_path  = os.path.join(model_dir, f'{architecture}_results.json')
    prof_path = os.path.join(model_dir, f'{architecture}_profiling.json')
    hist_path = os.path.join(model_dir, f'{architecture}_history.json')

    with open(res_path,  'w') as f: json.dump(results, f, indent=2)
    with open(hist_path, 'w') as f: json.dump(history, f, indent=2)
    with open(prof_path, 'w') as f:
        json.dump({
            'architecture': architecture,
            'split_protocol': '1_3_trainval__no_augmentation__80_20_val',
            'augmentation': False, 'config': cfg,
            'start_timestamp': model_start_ts, 'end_timestamp': model_end_ts,
            'total_time_s': round(model_total_s, 2), 'total_time_fmt': fmt_duration(model_total_s),
            'mean_fold_time_s': round(float(np.mean(fold_times)), 2),
            'mean_fold_time_fmt': fmt_duration(np.mean(fold_times)),
            'folds': profiling,
        }, f, indent=2)

    print(f'\n  {os.path.basename(res_path)}')
    print(f'  {os.path.basename(prof_path)}')
    print(f'  {os.path.basename(hist_path)}')
    return results


print('train_task4 (v3) carregada.')

---
## Treinamentos por Arquitetura

In [ ]:
results_t4_iv3 = train_task4('inceptionv3')

In [ ]:
results_t4_irv2 = train_task4('inceptionresnetv2')

In [ ]:
results_t4_xception = train_task4('xception')

In [ ]:
results_t4_effv2s = train_task4('efficientnetv2s')

---
## Célula 10 — Resumo Consolidado de Profiling

In [ ]:
ARCHS     = ['inceptionv3', 'inceptionresnetv2', 'xception', 'efficientnetv2s']
model_dir = os.path.join(MODELS_DIR, 'task4')
rows_perf = []; rows_time = []; rows_mem = []

for arch in ARCHS:
    res_path  = os.path.join(model_dir, f'{arch}_results.json')
    prof_path = os.path.join(model_dir, f'{arch}_profiling.json')
    if not os.path.exists(res_path):
        print(f'  {arch}: results não encontrado'); continue

    with open(res_path) as f: res = json.load(f)
    df_r  = pd.DataFrame(res)
    row_p = {'Arquitetura': arch}
    for m in ['precision','recall','accuracy','specificity','f1']:
        row_p[f'{m}_media']   = round(df_r[m].mean(), 4)
        row_p[f'{m}_desvpad'] = round(df_r[m].std(),  4)
    rows_perf.append(row_p)

    if not os.path.exists(prof_path): continue
    with open(prof_path) as f: prof = json.load(f)
    fold_times = [fd['time_fold_total_s'] for fd in prof['folds']]
    fold_ram   = [fd['ram_delta_mb']      for fd in prof['folds']]
    gpu_peaks  = [fd['gpu_after_train']['peak_mb'] for fd in prof['folds'] if fd.get('gpu_after_train')]

    rows_time.append({
        'Arquitetura': arch, 'Tempo total': prof['total_time_fmt'],
        'Tempo médio/fold': prof['mean_fold_time_fmt'],
        'Desvpad fold (s)': round(np.std(fold_times), 2),
        'Augmentation': prof.get('augmentation', 'N/A'),
        'Início': prof['start_timestamp'][:19], 'Fim': prof['end_timestamp'][:19],
    })
    rows_mem.append({
        'Arquitetura': arch,
        'Delta RAM médio (MB)': round(np.mean(fold_ram), 1),
        'Delta RAM max (MB)': round(max(fold_ram), 1),
        'GPU VRAM peak médio (MB)': round(np.mean(gpu_peaks), 1) if gpu_peaks else 'N/A',
        'GPU VRAM peak max (MB)':   round(max(gpu_peaks), 1)     if gpu_peaks else 'N/A',
    })

print('=' * 70)
print('TABELA 1 — Métricas de Classificação (μ ± σ, 5 folds)')
print('=' * 70)
if rows_perf: print(pd.DataFrame(rows_perf).set_index('Arquitetura').to_string())
print('\n' + '=' * 70)
print('TABELA 2 — Tempo de Execução')
print('=' * 70)
if rows_time: print(pd.DataFrame(rows_time).set_index('Arquitetura').to_string())
print('\n' + '=' * 70)
print('TABELA 3 — Consumo de Memória')
print('=' * 70)
if rows_mem: print(pd.DataFrame(rows_mem).set_index('Arquitetura').to_string())

if rows_perf: pd.DataFrame(rows_perf).to_csv(os.path.join(model_dir, 'summary_performance.csv'), index=False)
if rows_time: pd.DataFrame(rows_time).to_csv(os.path.join(model_dir, 'summary_timing.csv'),      index=False)
if rows_mem:  pd.DataFrame(rows_mem).to_csv( os.path.join(model_dir, 'summary_memory.csv'),      index=False)
print('\nCSVs consolidados salvos em models/task4/')

---
## Célula 11 — Resumo Final dos Resultados

In [ ]:
def print_results_table(results, task_name, arch_name):
    df = pd.DataFrame(results)
    print(f'\n{"─"*65}\n{task_name} | {arch_name}')
    print(df[['fold','precision','recall','accuracy','specificity','f1']].to_string(index=False))
    print('  Médias:')
    for col in ['precision','recall','accuracy','specificity','f1']:
        vals = df[col].values
        print(f'    {col:12s}: {vals.mean():.4f} ± {vals.std():.4f}')

print('=' * 65)
print('RESUMO FINAL — v3 (1/3 dataset, sem augmentation)')
print('=' * 65)

try:
    print_results_table(results_t4_iv3,      'Task 4', 'Inception-v3')
    print_results_table(results_t4_irv2,     'Task 4', 'InceptionResNet-v2')
    print_results_table(results_t4_xception, 'Task 4', 'Xception')
    print_results_table(results_t4_effv2s,   'Task 4', 'EfficientNetV2S')
except NameError:
    for arch in ARCHS:
        p = os.path.join(MODELS_DIR, 'task4', f'{arch}_results.json')
        if os.path.exists(p):
            with open(p) as f:
                print_results_table(json.load(f), 'Task 4', arch)